In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [ ]:
df = pd.read_csv("/content/combined_emotion.csv")   # path change kar
print(df.head())

                                            sentence emotion
0      i just feel really helpless and heavy hearted    fear
1  ive enjoyed being able to slouch about relax a...     sad
2  i gave up my internship with the dmrg and am f...    fear
3                         i dont know i feel so lost     sad
4  i am a kindergarten teacher and i am thoroughl...    fear


In [ ]:
df = df.dropna()
df['sentence'] = df['sentence'].astype(str)

In [ ]:
# Label Encoding
le = LabelEncoder()
df['emotion'] = le.fit_transform(df['emotion'])

In [ ]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    df['sentence'], df['emotion'], test_size=0.2, random_state=42
)

In [ ]:
# Tokenization
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq  = tokenizer.texts_to_sequences(X_test)

In [ ]:
# Padding
X_train_pad = pad_sequences(X_train_seq, maxlen=100)
X_test_pad  = pad_sequences(X_test_seq, maxlen=100)

In [ ]:
# Build Model LSTM
model = Sequential()
model.add(Embedding(5000, 64, input_length=100))
model.add(LSTM(64))
model.add(Dense(32, activation='relu'))
model.add(Dense(len(df['emotion'].unique()), activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
# Compile Model
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [ ]:
# Train Model
model.fit(X_train_pad, y_train, epochs=5, batch_size=64, validation_split=0.2)

Epoch 1/5
4228/4228 ━━━━━━━━━━━━━━━━━━━━ 348s 81ms/step - accuracy: 0.8903 - loss: 0.2506 - val_accuracy: 0.9367 - val_loss: 0.1091
Epoch 2/5
4228/4228 ━━━━━━━━━━━━━━━━━━━━ 379s 81ms/step - accuracy: 0.9380 - loss: 0.1008 - val_accuracy: 0.9372 - val_loss: 0.0995
Epoch 3/5
4228/4228 ━━━━━━━━━━━━━━━━━━━━ 386s 82ms/step - accuracy: 0.9406 - loss: 0.0931 - val_accuracy: 0.9381 - val_loss: 0.0970
Epoch 4/5
4228/4228 ━━━━━━━━━━━━━━━━━━━━ 394s 85ms/step - accuracy: 0.9428 - loss: 0.0887 - val_accuracy: 0.9377 - val_loss: 0.0974
Epoch 5/5
4228/4228 ━━━━━━━━━━━━━━━━━━━━ 342s 81ms/step - accuracy: 0.9438 - loss: 0.0859 - val_accuracy: 0.9320 - val_loss: 0.0985


In [ ]:
# Evaluate
loss, acc = model.evaluate(X_test_pad, y_test)
print("Accuracy:", acc)

2643/2643 ━━━━━━━━━━━━━━━━━━━━ 44s 17ms/step - accuracy: 0.9329 - loss: 0.0976
Accuracy: 0.9329391121864319
